In [1]:
import pyarrow.parquet as pq
import os
import pandas as pd

In [2]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
bookings_file = os.path.join(base_path, "bookings_single_line.parquet")
output_file = os.path.join(output_dir, "filtered_bookings.parquet")

In [3]:
# Columns to retain
keep_columns = [
    "book_stamp", "book_state", "booking_id", "serial_number_id",
    "workstep_number_mes", "station_id", "line_id", "part_group", "created_at"
]

In [4]:
# Step 1: Load the full Parquet file
table = pq.read_table(bookings_file)

In [5]:
# Step 2: Drop columns not in the keep list
columns_to_drop = [col for col in table.column_names if col not in keep_columns]
filtered_table = table.drop(columns_to_drop)

In [6]:
# Step 3: Save the filtered table
pq.write_table(filtered_table, output_file)
print(f"Filtered bookings saved to: {output_file}")

Filtered bookings saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_bookings.parquet


## Analyse columns

In [3]:
filtered_bookings_file = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered\filtered_bookings.parquet"

In [4]:
#Load filtered bookings
df = pd.read_parquet(filtered_bookings_file)

In [7]:
# Convert columns to datetime
df['book_stamp'] = pd.to_datetime(df['book_stamp'])
df['created_at'] = pd.to_datetime(df['created_at'])

# Calculate the time difference in seconds
df['time_diff'] = (df['book_stamp'] - df['created_at']).dt.total_seconds()

# Summary statistics of the difference
print(df['time_diff'].describe())

# Check how many records have zero difference
zero_diff_count = (df['time_diff'] == 0).sum()
print(f"Number of records with zero time difference: {zero_diff_count}")

count    1.446575e+06
mean     4.834948e-01
std      1.181255e+02
min      1.000000e-03
25%      1.100000e-02
50%      2.600000e-02
75%      5.300000e-02
max      5.807020e+04
Name: time_diff, dtype: float64
Number of records with zero time difference: 0


In [5]:
contingency_table = pd.crosstab(df['part_number'], df['part_group'])
print(contingency_table)

part_group   39d0d72e  42939de2  6db88e1e  7a78616d  83e223f1  8db45195  \
part_number                                                               
0cfa8302            0         0         0         0         0         0   
0e7994ba            0         0         0         0         0         0   
5a6867de            0         0         0         0         0     74057   
793c135e            0         6         0         0         0         0   
8852ee9e            0         0         0         0         0         0   
9d9d8f8a            0         0         0    511939         0         0   
a13143e7            0         0         0         0      6452         0   
b4044eb8            0         0         0         0         0         0   
d945376f            0         0     25065         0         0         0   
ec65f8bd       147408         0         0         0         0         0   

part_group   d5fd2a7a  f14cc036  f8d9094d  faa60612  
part_number                                  

In [6]:
# Calculate Cramér's V to measure association strength
def cramers_v(confusion_matrix):
    import numpy as np
    chi2 = scipy.stats.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))

In [8]:
import scipy.stats
cramers_v_value = cramers_v(contingency_table)
print(f"Cramér's V between part_number and part_group: {cramers_v_value:.4f}")

Cramér's V between part_number and part_group: 1.0000


In [4]:
# Overview of missing data (% of NaNs)
missing = df.isnull().mean().sort_values(ascending=False) * 100
print("\n=== Missing Values (% by column) ===")
print(missing)


=== Missing Values (% by column) ===
booking_id             0.0
book_state             0.0
workstep_number_mes    0.0
created_at             0.0
book_stamp             0.0
serial_number_id       0.0
station_id             0.0
part_number            0.0
part_group             0.0
erp_group_desc         0.0
line_id                0.0
dtype: float64


In [5]:
# Check cardinality (# unique values)
unique_counts = df.nunique().sort_values(ascending=False)
print("\n=== Cardinality (Number of Unique Values) ===")
print(unique_counts)


=== Cardinality (Number of Unique Values) ===
booking_id             1446345
book_stamp             1446169
created_at              887376
serial_number_id        309483
station_id                  11
part_number                 10
part_group                  10
erp_group_desc               8
workstep_number_mes          7
book_state                   3
line_id                      2
dtype: int64


In [6]:
# Top 10 most frequent values for object/categorical columns
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(f"\n=== {col} ===")
    print(f"Unique Values: {df[col].nunique()}")
    print("Top 10 frequent values:")
    print(df[col].value_counts().head(10))


=== booking_id ===
Unique Values: 1446345
Top 10 frequent values:
booking_id
57d25aad    2
c4e96162    2
5252b554    2
b6e5feb6    2
8860be19    2
1f449ad1    2
757ba31a    2
cb354060    2
01c5bc24    2
14afb20c    2
Name: count, dtype: int64

=== serial_number_id ===
Unique Values: 309483
Top 10 frequent values:
serial_number_id
d7273b00    14
43e70227    14
0a955706    14
e8102b55    13
8dc19bcd    11
ad0d3da9    11
1b56f7fb    11
b4ca18cf    11
c6037fa2    11
1e0814c0    11
Name: count, dtype: int64

=== station_id ===
Unique Values: 11
Top 10 frequent values:
station_id
a78230b0    210955
904d6ebc    206003
450f2075    205803
2435c59b    205091
bbee591d    205062
6ecd370a    204864
c61cc047    103217
38b291ac     27115
1405e64b     26962
0c3fb4ee     25788
Name: count, dtype: int64

=== part_number ===
Unique Values: 10
Top 10 frequent values:
part_number
9d9d8f8a    511939
8852ee9e    430133
0e7994ba    171702
ec65f8bd    147408
5a6867de     74057
b4044eb8     41710
0cfa8302     

In [7]:
# Basic info to check data types & memory usage
print("\n=== Dataset Info ===")
print(df.info())


=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1446575 entries, 0 to 1446574
Data columns (total 11 columns):
 #   Column               Non-Null Count    Dtype              
---  ------               --------------    -----              
 0   booking_id           1446575 non-null  object             
 1   book_state           1446575 non-null  int32              
 2   workstep_number_mes  1446575 non-null  int32              
 3   created_at           1446575 non-null  datetime64[us, UTC]
 4   book_stamp           1446575 non-null  datetime64[us, UTC]
 5   serial_number_id     1446575 non-null  object             
 6   station_id           1446575 non-null  object             
 7   part_number          1446575 non-null  object             
 8   part_group           1446575 non-null  object             
 9   erp_group_desc       1446575 non-null  object             
 10  line_id              1446575 non-null  object             
dtypes: datetime64[us, UTC](2), i